# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [16]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [17]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'resume page', 'url': 'https://edwarddonner.com/curriculum/'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 7 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'product page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'company page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [18]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [19]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
8 days ago
•
535k
•
3.52k
XiaomiMiMo/MiMo-V2.5-Pro
Updated
7 days ago
•
11.8k
•
424
openai/privacy-filter
Updated
12 days ago
•
133k
•
1.25k
mistralai/Mistral-Medium-3.5-128B
Updated
about 5 hours ago
•
12k
•
253
talkie-lm/talkie-1930-13b-it
Updated
11 days ago
•
223
Browse 2M+ models
Spaces
Running
on
Zero
MCP
2.53k
Wan2.2 14B Preview
🐌
2.53k
generate a video from an image with a text prompt
Running
on
Zero
MCP
919
Wan2.2 14B Fast Preview
🐌
919
generate a video from an image with a text prompt
Running
on
CPU Upgrade
Featured
291
ML Intern
🤖
291
Chat with an ML Intern to get machine‑learning 

In [20]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [21]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [22]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4-Pro\nUpdated\n8 days ago\n•\n535k\n•\n3.52k\nXiaomiMiMo/MiMo-V2.5-Pro\nUpdated\n7 days ago\n•\n11.8k\n•\n424\nopenai/privacy-filter\nUpdated\n12 days ago\n•\n133k\n•\n1.25k\nmistralai/Mistral-Medium-3.5-128B\nUpdated\nabout 5 hours ago\n•\n12k\n•\n253\ntalkie-lm/talkie-1930-13b-it\nUpdated\n11 days ago\n•\n223\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nMCP\n2.

In [25]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [26]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## Who We Are

Hugging Face is the premier collaboration platform for the global machine learning (ML) community. We empower ML engineers, scientists, and enthusiasts to share, discover, and build upon open-source machine learning models, datasets, and applications. Our mission is to foster an open, ethical AI future by connecting a fast-growing community with cutting-edge technology.

Join millions of users exploring over **2 million models**, **500k+ datasets**, and **1 million+ applications** spanning text, image, video, audio, and 3D modalities.

---

## What We Offer

### The Hugging Face Hub
- Centralized platform to **host, share, and collaborate** on ML models, datasets, and applications.
- Open-source tools and libraries that accelerate ML research and deployment.
- Build your professional ML profile by sharing your projects with a vibrant, global community.

### Enterprise Solutions
Hugging Face provides advanced AI platforms tailored for organizations:
- **Team Plans** starting at $20/user/month for collaborative development.
- **Enterprise Plans** with features like:
  - Enterprise-grade security and access controls.
  - Single Sign-On (SSO) for seamless, secure user management.
  - Granular resource groups and token management.
  - Comprehensive audit logs for compliance.
  - Dedicated support and analytics dashboards.
  - Scalable compute options including ZeroGPU for high-performance needs.
  - Private datasets viewer and additional private storage.

---

## Company Culture

At Hugging Face, we embrace **transparency, collaboration, and diversity**. We believe in the power of shared knowledge and open innovation to drive the AI revolution. Our talented science and development teams work at the forefront of machine learning, fostering an environment where curiosity and ethical AI development thrive.

We encourage researchers, engineers, and users at all levels to contribute and collaborate through our platform and community channels — including GitHub, Discord, and Twitter.

---

## Join Us

Are you passionate about AI and machine learning? Hugging Face is continuously growing and looking for talented individuals in research, engineering, data science, and community management. By joining us, you will:

- Work alongside top experts pushing the boundaries of AI.
- Influence tools used by millions worldwide.
- Contribute to open-source projects that shape the future.
- Enjoy a culture of inclusivity, learning, and innovation.

Explore current career opportunities and become part of the AI community shaping tomorrow.

---

## Our Customers & Community

Hugging Face serves a broad spectrum of customers:
- Individual ML researchers and hobbyists leveraging our open platform.
- Educational institutions aiming to enhance AI curriculum and projects.
- Startups innovating with AI-powered products.
- Enterprises seeking secure, scalable AI infrastructure with dedicated support.

Join a global community that includes top AI labs, industry leaders, and passionate practitioners who trust Hugging Face to accelerate machine learning innovation.

---

## Get Started Today!

- Explore over **2 million ML models** and **500k datasets** for free.
- Share your work and build your portfolio.
- Leverage robust enterprise tools to scale your AI projects.

**Visit:** [huggingface.co](https://huggingface.co)  
**Follow us:** GitHub | Twitter | LinkedIn | Discord

---

### Hugging Face Colors & Brand

Our signature brand colors reflect our vibrant community spirit:  
- Yellow: #FFD21E  
- Orange: #FF9D00  
- Gray: #6B7280  

Embodying energy, innovation, and reliability.

---

Hugging Face — The AI community building the future.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [27]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a pioneering AI company and a vibrant community dedicated to building the future of machine learning. It serves as a central collaboration platform where the global machine learning community comes together to create, share, and experiment with models, datasets, and applications. Hugging Face empowers engineers, scientists, and enthusiasts to innovate and accelerate their work while fostering an open and ethical AI ecosystem.

---

## What We Offer

- **Hugging Face Hub:** The core platform hub where users can host, discover, and collaborate on *over 2 million* open-source machine learning models and *500,000+ datasets* across various modalities including text, image, audio, video, and even 3D.  
- **Spaces:** Interactive ML applications running live on the platform, ranging from video generation to image editing and advanced 3D content creation. Users can explore and deploy AI apps efficiently.  
- **Open Source Stack:** An open-source machine learning stack that enables faster development and deployment of ML projects fostering innovation through community collaboration.  
- **Enterprise Solutions:** Customizable and scalable AI solutions tailored for businesses seeking to integrate advanced machine learning capabilities.  
- **Paid Plans:** Accelerate your ML projects with priority access to resources and enhanced tools designed for professional and enterprise users.

---

## Company Culture

At Hugging Face, the culture revolves around **openness, collaboration, and ethical AI development**:  
- **Community-Driven:** The company is deeply invested in supporting a fast-growing, global community of researchers, developers, and enthusiasts who contribute to and benefit from shared resources.  
- **Inclusive Innovation:** Everyone—from ML interns to expert scientists—can contribute, learn, grow, and build their machine learning portfolio through the platform.  
- **Ethics and Openness:** Hugging Face commits to creating an open and ethical AI future where transparency and responsible AI are paramount.  
- **Encouraging Growth:** The platform offers opportunities to engage with cutting-edge AI applications and datasets, advancing users' skills and careers.

---

## Our Customers & Community

Hugging Face serves a diverse community including:  
- **Machine Learning Researchers & Engineers:** Access to cutting edge models and datasets to accelerate research and production workflows.  
- **Developers & Startups:** Tools and infrastructure to build, deploy, and share ML-powered applications quickly.  
- **Enterprises:** Scalable AI solutions integrated into business processes for innovation and competitive advantage.  
- **Academic & Hobbyist Users:** Opportunities to learn, experiment, and contribute in an open and supportive environment.

The platform hosts trending models daily, such as *DeepSeek-V4-Pro*, *MiMo-V2.5-Pro*, and *Mistral-Medium*, illustrating its role as the hub for state-of-the-art AI models and applications.

---

## Careers at Hugging Face

Joining Hugging Face means being part of a mission-driven team passionate about pushing the boundaries of machine learning technology. The company invests in nurturing talent through:  
- **Collaborative Environment:** Work alongside a talented, diverse team contributing to open science and AI ethics.  
- **Growth Opportunities:** Engage with cutting-edge projects and the wider ML community to grow your skills and visibility.  
- **Internships and Roles:** From ML interns to senior positions, Hugging Face offers multiple career entry points focused on technological innovation and community impact.  

Explore opportunities and become part of a company shaping the AI community and its future.

---

## Get Started

- **Explore 2M+ models and 500k+ datasets** on the [Hugging Face Hub](https://huggingface.co).  
- **Build and share your ML portfolio** while collaborating with a global, active AI community.  
- **Accelerate your projects** with open-source tools or opt for enterprise-grade solutions designed for scale and impact.

**Join the AI community building the future — at Hugging Face.**

---

*For more information, visit:*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>